# The Jam With No Cause

### An agent-based model of phantom traffic jams

Cars travel down a one-lane road. There is no accident, no traffic light, no
merge, and no bad driver. Every car wants to keep moving, every car follows
exactly the same rule, and traffic still comes to a halt.

This notebook is the computational companion to the paper worksheet. It is
organised so that you can see the same model at three scales:

1. **The paper scale.** Twelve cells, five cars, fifteen ticks. The code
   reproduces the worked example from the worksheet exactly, digit for digit,
   so you can check the program against something you did by hand.
2. **The visual scale.** A hundred cells and a couple of hundred ticks, plotted
   as a space-time diagram. This is where you can *see* the jam travelling
   backwards while every car moves forwards.
3. **The statistical scale.** Thousands of runs across every possible density,
   which is how you find the threshold that the worksheet asks students to
   guess at.

Everything runs on `numpy` and `matplotlib`. There is nothing to install.

**How to read the code.** The comments are deliberately heavy, and they are
aimed at the modelling decisions rather than at the Python. Whenever a line of
code encodes an assumption about the world — rather than a fact about the
world — the comment says so. Those are the lines a critic of the model would
attack, and they are the lines students should learn to notice.

## 1. The model in words

Before any code, here is the whole model. If you cannot state a model this
compactly, you probably do not understand it yet.

**Environment.** A ring of `L` cells. Traffic flows in the direction of
increasing index, and the last cell leads back around to the first. A cell
holds either one car or nothing. The ring is a convenience, not a claim about
real roads: it lets us study a fixed number of cars without modelling what
happens at the ends of the road.

**Agents.** Each car is identical. A car can see exactly one cell ahead of
itself and nothing else — not the cell beyond that, not the cell behind it,
not the state of the road as a whole. A car has no memory and no plan.

**States.** Each tick, a car ends up either *moving* or *stopped*. This is an
output of the rules, not an input. Nothing in the model stores it between
ticks.

**Rules.** Applied to every car simultaneously, from the same snapshot of the
road:

1. If the cell directly ahead is empty, move into it.
2. If the cell directly ahead holds a car, stay put.
3. *The brake.* Some cars are designated as braking this tick. A braking car
   stays put even if the road ahead of it is clear.

That is all. Three rules, no parameters other than the length of the road, the
number of cars, and how often the brake fires.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A road is represented as a 1-D boolean array of length L.
#   road[i] == True   ->  cell i holds a car
#   road[i] == False  ->  cell i is empty
#
# Note what this representation does NOT store: any per-car identity, speed,
# destination, driver personality, or history. That is a modelling choice, and
# it is only legitimate because the rules treat every car identically. The
# moment you want a slow truck or an impatient driver, this representation has
# to change.
#
# Cells are 0-indexed here, while the worksheet numbers them 1 to 12. The
# printing helper below converts, so the on-screen grids match the paper.

L_PAPER = 12                       # road length used on the worksheet
CARS_A = [1, 2, 3, 7, 9]           # Run A start, in worksheet (1-based) cells
CARS_B = [1, 5, 9]                 # Run B start
CARS_C = [1, 2, 3, 4, 7, 8, 10, 11]  # Run C start


def make_road(length, occupied_cells_1based):
    """Build a road with cars in the given 1-based cells."""
    road = np.zeros(length, dtype=bool)
    for c in occupied_cells_1based:
        road[c - 1] = True
    return road

## 2. The rule, in code

This is the entire model. Everything else in the notebook is scaffolding for
running it, drawing it, or measuring it.

Two details in this function do most of the work, and neither of them is
visible in the three-line statement of the rules above.

**Simultaneous updating.** We build a brand-new road array and read decisions
only from the old one. If instead we modified the road in place, a car would
sometimes see a gap that its predecessor opened *during the tick we are
currently computing*, and the jams would largely dissolve. Nothing in the rules
tells us which of these to do. The choice is ours, it changes the result, and
it has to be declared. Section 8 measures how much it matters.

**Who brakes is an input, not part of the rule.** The function takes the set of
braking cells as an argument rather than rolling dice internally. That keeps
the deterministic mechanics separate from the stochastic input, which means we
can feed in the exact dice rolls from the worksheet to check the code, and feed
in random ones when we want statistics.

In [ ]:
def step(road, braking=()):
    """Advance the road by one tick and return (new_road, n_stopped).

    Parameters
    ----------
    road : boolean array
        The current state of the road. This array is never modified.
    braking : iterable of int
        0-based indices of cells whose car (if any) refuses to move this tick.

    The returned n_stopped counts cars that did not move, for any reason —
    blocked or braking. We do not distinguish the two, because from the road's
    point of view a stopped car is a stopped car.
    """
    length = len(road)

    # Fresh array. Every decision below is read from `road` (the old snapshot)
    # and written to `new_road`. This is what makes the update simultaneous.
    new_road = np.zeros(length, dtype=bool)

    braking = set(braking)
    n_stopped = 0

    for i in range(length):
        if not road[i]:
            continue                      # empty cell: nothing to decide

        ahead = (i + 1) % length          # the modulo is the ring wrap-around

        # Rule 2 (blocked) and rule 3 (braking) have the same consequence, so
        # they collapse into one branch. Rule 3 is checked here rather than
        # earlier because it only ever *prevents* motion; it can never cause it.
        if road[ahead] or i in braking:
            new_road[i] = True
            n_stopped += 1
        else:
            # Rule 1. This write can never collide with another car: cell
            # `ahead` was empty in the old snapshot, so no car was sitting
            # there to stay put, and the only car that can move into it is
            # this one.
            new_road[ahead] = True

    return new_road, n_stopped

## 3. Two ways of choosing who brakes

The worksheet has students roll a die twice to pick one cell out of twelve. If
a car is sitting in that cell it brakes; if the cell is empty, nobody brakes
this tick. That is a slightly odd-looking mechanism, and it was chosen for a
specific reason: it gives **every car the same 1-in-12 chance of braking, in
every run, regardless of how many cars are on the road**. Students therefore
cannot explain away a jammed run by saying the drivers were worse.

For larger roads that mechanism gets diluted — one braking cell out of a
hundred is a much gentler disturbance than one out of twelve — so the second
function lets each car brake independently with probability `p`. On a
twelve-cell road with `p = 1/12` the two are equivalent in expectation, which
is why we can use the first to check the code against the paper and the second
for everything else.

In [ ]:
def brake_one_cell(length, rng):
    """Worksheet mechanism: pick exactly one cell of the road at random.

    Returns a tuple so it can be passed straight to `step`. The chosen cell is
    returned whether or not it contains a car — `step` simply finds nothing to
    stop if the cell is empty. Per-car brake probability is 1/length.
    """
    return (int(rng.integers(length)),)


def brake_independently(road, p, rng):
    """Each car brakes on its own with probability p, independent of the rest.

    Note the asymmetry with the function above: this one needs to know where
    the cars are, because the probability attaches to cars rather than to
    cells. Both are defensible models of "drivers occasionally dab the brake
    for no reason", and neither is more correct than the other. They are
    different assumptions, and they are worth being explicit about.
    """
    occupied = np.flatnonzero(road)
    return tuple(int(i) for i in occupied if rng.random() < p)

## 4. Checking the code against the paper

This is the most important cell in the notebook, and it is the step most people
skip.

The worksheet works through three ticks by hand with specific dice rolls. If
the program is a faithful implementation, it must produce those exact rows. So
we feed in the same rolls and assert the results. If an assertion fires, the
code is wrong — or, just as usefully, the worksheet is wrong.

Getting into the habit of validating a simulation against a hand-computed case
is worth more than any amount of downstream cleverness. A model that produces
beautiful plots and disagrees with arithmetic is not a model, it is an
illustration.

The worksheet's rolls: tick 1 hits cell 3, tick 2 hits cell 12 (empty), tick 3
hits cell 5 (empty).

In [ ]:
# Worksheet dice rolls, converted from 1-based cells to 0-based indices.
paper_rolls = [3, 12, 5]

# Expected rows, straight from the worksheet and its facilitator notes.
expected = [
    ([1, 2, 3, 8, 10], 3),    # after tick 1
    ([1, 2, 4, 9, 11], 2),    # after tick 2
    ([1, 3, 5, 10, 12], 1),   # after tick 3 (the practice row)
]

road = make_road(L_PAPER, CARS_A)
for tick, (roll_1based, (want_cells, want_stopped)) in enumerate(
        zip(paper_rolls, expected), start=1):
    road, stopped = step(road, braking=(roll_1based - 1,))
    got_cells = [int(i) + 1 for i in np.flatnonzero(road)]

    assert got_cells == want_cells, f"tick {tick}: {got_cells} != {want_cells}"
    assert stopped == want_stopped, f"tick {tick}: {stopped} != {want_stopped}"
    print(f"tick {tick}: cars in cells {got_cells}, {stopped} stopped  OK")

print("\nThe code reproduces the worksheet exactly.")

## 5. Running and printing a whole run

`simulate` just calls `step` in a loop and keeps every row. Note that it
returns the full history rather than a summary: in agent-based modelling the
trajectory usually *is* the result, and collapsing it to a mean is often how
people manage to miss the interesting behaviour entirely.

`show_grid` prints a run in the same layout as the worksheet, so a group can
check their pencil grid against the machine.

In [ ]:
def simulate(road, n_ticks, rng, brake=None):
    """Run the model and return (history, stopped_per_tick).

    history has shape (n_ticks + 1, L); row 0 is the starting arrangement, so
    there is no stopped count for it.

    `brake` is a callable taking (road, rng) and returning braking cells. The
    default reproduces the worksheet mechanism.
    """
    if brake is None:
        brake = lambda r, g: brake_one_cell(len(r), g)

    history = [road.copy()]
    stopped = []
    for _ in range(n_ticks):
        road, n = step(road, braking=brake(road, rng))
        history.append(road.copy())
        stopped.append(n)
    return np.array(history), np.array(stopped)


def show_grid(history, stopped, title=""):
    """Print a run in worksheet layout: X for a car, dot for an empty cell."""
    length = history.shape[1]
    if title:
        print(title)
    print("      " + " ".join(f"{c + 1:>2}" for c in range(length))
          + "   stopped")
    for t, row in enumerate(history):
        marks = " ".join(" X" if v else "  ." for v in row)
        tail = "  -" if t == 0 else f"{stopped[t - 1]:>3}"
        print(f"  {t:>2}  {marks}    {tail}")
    if len(stopped):
        print(f"\n  total stopped-car count: {stopped.sum()}")

In [ ]:
# A seeded generator so the notebook is reproducible. Change the seed and you
# get a different run, which is exactly why the cell after Run C repeats
# each density hundreds of times instead of trusting one grid.
rng = np.random.default_rng(7)

hist_a, stop_a = simulate(make_road(L_PAPER, CARS_A), 15, rng)
show_grid(hist_a, stop_a, "Run A - five cars, 15 ticks")

### Runs B and C: changing only the number of cars

Same road, same three rules, same per-car brake probability. The only thing
that changes is how many cars are on the road.

In [ ]:
rng = np.random.default_rng(7)
hist_b, stop_b = simulate(make_road(L_PAPER, CARS_B), 12, rng)
show_grid(hist_b, stop_b, "Run B - three cars, 12 ticks")

print()
rng = np.random.default_rng(7)
hist_c, stop_c = simulate(make_road(L_PAPER, CARS_C), 12, rng)
show_grid(hist_c, stop_c, "Run C - eight cars, 12 ticks")

In [ ]:
# Stopped cars per tick, expressed as a fraction of the cars present, so the
# three runs are comparable. A raw count would obviously be larger with more
# cars; the question is whether each individual car is worse off.
for name, cars, stop in [("Run B (3 cars)", 3, stop_b),
                         ("Run A (5 cars)", 5, stop_a),
                         ("Run C (8 cars)", 8, stop_c)]:
    frac = stop.mean() / cars
    print(f"{name}:  {stop.sum():>3} stopped-car-ticks   "
          f"{frac:.0%} of cars stopped in an average tick")

A single run of fifteen ticks is a sample of size one from a stochastic
process, so do not over-read these three numbers. The next cell does what the
worksheet cannot: repeats each run several hundred times and looks at the
distribution.

This is the honest reason agent-based models are run in bulk. It is not that
one run is uninformative — it is that one run cannot tell you whether what you
saw was the rule or the dice.

In [ ]:
def repeat_runs(start_cells, n_ticks, n_reps, seed=0, length=L_PAPER):
    """Return the mean stopped-fraction from each of n_reps independent runs."""
    rng = np.random.default_rng(seed)
    n_cars = len(start_cells)
    out = np.empty(n_reps)
    for r in range(n_reps):
        _, stopped = simulate(make_road(length, start_cells), n_ticks, rng)
        out[r] = stopped.mean() / n_cars
    return out


reps = {name: repeat_runs(cells, 15, 400)
        for name, cells in [("3 cars", CARS_B),
                            ("5 cars", CARS_A),
                            ("8 cars", CARS_C)]}

for name, vals in reps.items():
    print(f"{name}:  mean {vals.mean():.1%}   "
          f"middle 90% of runs {np.percentile(vals, 5):.0%}"
          f" to {np.percentile(vals, 95):.0%}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist([reps["3 cars"], reps["5 cars"], reps["8 cars"]], bins=14,
        label=list(reps.keys()))
ax.set_xlabel("fraction of cars stopped in an average tick")
ax.set_ylabel("runs")
ax.set_title("400 runs of each density, 15 ticks each")
ax.legend()
plt.tight_layout()
plt.show()

Notice that the three-car and five-car distributions overlap a good deal, while
the eight-car distribution sits well clear of both. On a fifteen-tick run at
low density, luck genuinely can dominate — which is a reasonable thing for
students to discover, and a reasonable defence if a group's pencil grid looks
nothing like their neighbour's.

## 6. The picture: a jam that travels backwards

Twelve cells is too cramped to see the central phenomenon. Here is an
eighty-cell road drawn as a **space-time diagram**: each row is one tick, time
runs downwards, position runs rightwards, and a dark mark is a car.

The thin diagonal streaks running down and to the right are individual cars
making progress. The dark clumps are jams. Follow a clump down the page and it
drifts to the *left* — upstream, against the direction of travel — while every
car in the picture moves right.

Be warned that this particular model makes the effect hard to see by eye, for a
reason worth understanding. Cars here have one speed, so in free flow they
advance a bit under one cell per tick, and jam fronts eat backwards at about
one cell per tick. The two are nearly mirror images, so the trajectories and
the jam edges sit at similar angles and the picture is busy. Section 10 gives
cars a top speed, which separates the two scales and produces the textbook
version of this plot. For now the honest move is to stop squinting and measure
it, which is the next cell.

In [ ]:
L_BIG = 80
DENSITY = 0.28
P_BRAKE = 0.18
N_TICKS = 400          # long run, for the measurement below
N_SHOWN = 70           # but only the first stretch is legible when plotted

rng = np.random.default_rng(11)

# Scatter the cars at random rather than in a block, so that nobody can claim
# the jams below are left over from a contrived starting arrangement. They form
# on their own, from a uniform road and nothing else.
n_cars = int(L_BIG * DENSITY)
start_big = np.zeros(L_BIG, dtype=bool)
start_big[rng.choice(L_BIG, size=n_cars, replace=False)] = True

hist_big, stop_big = simulate(
    start_big.copy(), N_TICKS, rng,
    brake=lambda r, g: brake_independently(r, P_BRAKE, g))

fig, ax = plt.subplots(figsize=(9, 7.5))
ax.imshow(hist_big[:N_SHOWN + 1], aspect="auto", interpolation="nearest",
          cmap="binary")
ax.set_xlabel("position along the road (cell)")
ax.set_ylabel("time (tick)")
ax.set_title(f"{n_cars} cars on {L_BIG} cells, brake probability {P_BRAKE}")
plt.tight_layout()
plt.show()

### Measuring the backward drift

Eyeballing a diagonal is not measurement, so here is a measurement.

A cell counts as **jammed** at tick *t* if it holds a car at both *t* and
*t+1* — that is, a car sat there and failed to move. That gives us a picture of
where the jams are at every tick. To find out how fast the jams travel, we ask
a simple question: by how many cells do we have to shift tick *t+1*'s jam
pattern to make it line up best with tick *t*'s?

This trick is worth adding to your toolkit. It works on any spatial pattern
whose speed you want to estimate and it needs no notion of individual jam
identity, which matters here because jams merge, split, and die constantly.
Trying to track "the same jam" over time would require arbitrary decisions
about when two jams are one; the alignment approach sidesteps that entirely.

In [ ]:
def jam_alignment_scores(history, warmup=100, max_shift=4):
    """How well do consecutive jam patterns line up at each spatial shift?

    A high score at shift d means the jam pattern moved d cells between ticks.
    Negative d is upstream, against the traffic.

    We discard the first `warmup` ticks because the early road still carries
    the fingerprint of the random starting arrangement.
    """
    jammed = np.array([history[t] & history[t + 1]
                       for t in range(len(history) - 1)])[warmup:]

    scores = {}
    for d in range(-max_shift, max_shift + 1):
        total = 0
        for t in range(len(jammed) - 1):
            # np.roll shifts the pattern around the ring, which is exactly the
            # right thing on a circular road: no edge cases to special-case.
            total += int(np.sum(jammed[t] & np.roll(jammed[t + 1], -d)))
        scores[d] = total
    return scores


scores = jam_alignment_scores(hist_big)
best = max(scores, key=scores.get)

for d, v in scores.items():
    bar = "#" * int(40 * v / max(scores.values()))
    mark = "  <-- best" if d == best else ""
    print(f"shift {d:+d} cells: {v:>5}  {bar}{mark}")

mean_car_speed = 1 - stop_big.mean() / n_cars
print(f"\njams travel   {best:+d} cells per tick")
print(f"cars travel   {mean_car_speed:+.2f} cells per tick on average")
print("\nOpposite signs. The jam goes one way, the cars go the other.")

The alignment is sharpest at a shift of one cell upstream. Zero comes second,
which is fair enough — jams do partly sit still while they exist. The
informative number is the other one: forward motion scores lower than any other
shift on the table, by a wide margin. Whatever the jams are doing, they are
never doing it in the direction the cars are going.

There is a tidy reason for the best shift being exactly one. When the car at the very back of a jam finally
pulls away, the car behind it does not find out until the following tick,
because it only ever looks one cell ahead. The tail of the jam therefore
consumes the queue behind it at precisely one cell per tick, no matter what the
cars at the front of the jam are doing.

So the jam is not a thing that moves. It is a *position in a queue* that gets
handed backwards, one car per tick, like a rumour.

## 7. Finding the threshold

The worksheet asks a fast-finishing group to hunt for the density at which
brakes stop washing out and start accumulating. Here we can just sweep every
density and look.

The quantity to plot is **flow**: the average number of cars crossing any given
point per tick, which for this rule equals density times the fraction of cars
moving. Flow is what a traffic engineer cares about, and it behaves
non-monotonically in a way that catches most people out.

In [ ]:
def measure_flow(length, n_cars, p, rng, n_ticks=400, warmup=100):
    """Average flow, discarding an initial transient.

    The warmup matters. The first several dozen ticks still carry the
    fingerprint of the random starting arrangement, and including them biases
    the estimate. Deciding how long a transient lasts is a judgement call in
    every simulation study; here 100 ticks is comfortably enough, which you can
    verify by plotting the running mean.
    """
    if n_cars == 0:
        return 0.0
    road = np.zeros(length, dtype=bool)
    road[rng.choice(length, size=n_cars, replace=False)] = True

    moved_total = 0
    for t in range(n_ticks):
        road, stopped = step(road, brake_independently(road, p, rng))
        if t >= warmup:
            moved_total += n_cars - stopped
    return moved_total / ((n_ticks - warmup) * length)


rng = np.random.default_rng(1)
L_SWEEP = 100
car_counts = np.arange(0, L_SWEEP + 1, 2)

curves = {}
for p in [0.0, 0.10, 0.30]:
    curves[p] = np.array([measure_flow(L_SWEEP, int(n), p, rng)
                          for n in car_counts])

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for p, flow in curves.items():
    label = "no brake (p = 0)" if p == 0 else f"brake probability {p}"
    ax.plot(car_counts / L_SWEEP, flow, marker="o", ms=3, label=label)
ax.axvline(0.5, ls=":", lw=1, color="grey")
ax.text(0.505, ax.get_ylim()[1] * 0.05, " density 0.5", fontsize=8,
        color="grey")
ax.set_xlabel("density (cars per cell)")
ax.set_ylabel("flow (cars per cell per tick)")
ax.set_title("Flow against density")
ax.legend()
plt.tight_layout()
plt.show()

Three things are worth pulling out of that plot.

**The curve turns over.** Adding cars raises flow up to a peak and lowers it
afterwards. Past the peak the road carries *less* traffic than it would with
fewer cars on it. This is the single most counterintuitive fact about traffic,
and it fell out of three rules and a die.

**The peak sits near density 0.5.** With no brake the model is exactly solvable:
below half-full every car can eventually be moving, so flow equals density;
above half-full there are not enough gaps to go round, so flow equals one minus
density. That gives a perfect triangle peaking at 0.5. Scale that back to the
worksheet's twelve cells and the peak is at six cars — which is exactly the
threshold the facilitator notes tell you to expect students to find.

**The brake only bites in the middle.** At very low density there is room to
absorb a brake; at very high density the road is jammed anyway and the brake
adds nothing. In between, one car's twitch propagates. The disturbance and the
density are not independent causes that add up — the density decides whether
the disturbance matters at all.

## 8. Question 6: what if every driver were perfect?

Delete rule 3 entirely. No brakes, no randomness, no bad drivers — just cars
that advance the instant the road ahead of them is clear.

Run C on the worksheet has eight cars on twelve cells, a density of two thirds.
Watch what perfect driving achieves.

In [ ]:
road = make_road(L_PAPER, CARS_C)
history = [road.copy()]
stopped_seq = []
for _ in range(12):
    road, n = step(road, braking=())     # rule 3 removed
    history.append(road.copy())
    stopped_seq.append(n)

show_grid(np.array(history), np.array(stopped_seq),
          "Run C with the brake deleted - eight perfect drivers")

The road settles into a permanent pattern in which four of the eight cars are
stopped every single tick, forever. Not because anyone braked, but because
eight cars cannot all have an empty cell in front of them on a twelve-cell
road. The arithmetic forbids it.

This is worth dwelling on with students. The intuition that jams are caused by
bad driving is not merely incomplete; above a certain density it is
unavailable. There is no behavioural intervention that produces free flow,
because the constraint is geometric. The only remedies are fewer cars or more
road.

In [ ]:
# The same point across all densities: the best achievable flow with perfect
# drivers, compared with what the road would carry if every car always moved.
rng = np.random.default_rng(3)
dens = np.linspace(0.02, 0.98, 25)
perfect = [measure_flow(100, int(round(d * 100)), 0.0, rng) for d in dens]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dens, perfect, marker="o", ms=3, label="perfect drivers, no brake")
ax.plot(dens, dens, ls="--", lw=1, label="if every car always moved")
ax.set_xlabel("density")
ax.set_ylabel("flow")
ax.set_title("Even flawless drivers cannot beat the geometry")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Question 7: the choice nobody told you about

The worksheet insists that every car's move be decided from the same snapshot
of the road. Here is the alternative, and it is the one most people implement
by accident: walk along the road updating cars in place, so that a car sees the
gap the car in front has just opened.

On a ring there is no "front of the queue", so we have to pick a cell to start
from. That arbitrariness is part of the point.

In [ ]:
def step_sequential(road, braking=(), start=None, rng=None):
    """Advance one tick by updating cars in place, front to back.

    Contrast with `step`: this modifies a copy of the road as it goes, so a car
    can move into a cell that the car ahead of it vacated *within this same
    tick*. Information therefore travels an unbounded distance downstream in
    one tick, which is a strong and largely unexamined assumption about how
    fast drivers react.

    We iterate from `start` backwards so that cars further along the road are
    resolved first. On a ring, `start` is arbitrary; randomising it at least
    avoids baking in a fixed privileged position.
    """
    length = len(road)
    new_road = road.copy()
    braking = set(braking)
    n_stopped = 0

    if start is None:
        start = 0 if rng is None else int(rng.integers(length))

    for k in range(length):
        i = (start - k) % length          # walk upstream from `start`
        if not new_road[i]:
            continue
        ahead = (i + 1) % length
        if new_road[ahead] or i in braking:
            n_stopped += 1
        else:
            new_road[i] = False
            new_road[ahead] = True

    return new_road, n_stopped

In [ ]:
# Question 7 as the worksheet poses it: Run C, twelve cells, eight cars. We run
# it 500 times under each update scheme, because a single 12-tick run is far
# too noisy to compare schemes.
def repeat_scheme(sequential, n_reps=500, n_ticks=12):
    out = np.empty(n_reps)
    for rep in range(n_reps):
        rng = np.random.default_rng(rep)      # same seeds for both schemes
        road = make_road(L_PAPER, CARS_C)
        total = 0
        for _ in range(n_ticks):
            braking = brake_one_cell(L_PAPER, rng)
            if sequential:
                road, n = step_sequential(road, braking, rng=rng)
            else:
                road, n = step(road, braking)
            total += n
        out[rep] = total / n_ticks / len(CARS_C)
    return out


sim = repeat_scheme(sequential=False)
seq = repeat_scheme(sequential=True)
print("Run C, 500 repetitions of each scheme")
print(f"  simultaneous update: {sim.mean():.0%} of cars stopped per tick")
print(f"  sequential update:   {seq.mean():.0%} of cars stopped per tick")

Roughly 57 percent against 26 percent. Changing nothing but the order in which
you resolve the cars more than halves the amount of stopping. On the crowded
twelve-cell road the two schemes are not variants of one model; they are
different models with different answers.

The effect is milder on a longer, emptier road, but it does not go away.

In [ ]:
def simulate_sequential(road, n_ticks, rng, p):
    history = [road.copy()]
    stopped = []
    for _ in range(n_ticks):
        road, n = step_sequential(
            road, braking=brake_independently(road, p, rng), rng=rng)
        history.append(road.copy())
        stopped.append(n)
    return np.array(history), np.array(stopped)


# Identical starting road, identical brake probability, identical seed.
hist_sim, stop_sim = simulate(
    start_big.copy(), N_TICKS, np.random.default_rng(99),
    brake=lambda r, g: brake_independently(r, P_BRAKE, g))
hist_seq, stop_seq = simulate_sequential(
    start_big.copy(), N_TICKS, np.random.default_rng(99), P_BRAKE)

print(f"{L_BIG} cells, {n_cars} cars")
print(f"  simultaneous: {stop_sim.mean() / n_cars:.1%} stopped per tick")
print(f"  sequential:   {stop_seq.mean() / n_cars:.1%} stopped per tick")

fig, axes = plt.subplots(1, 2, figsize=(12, 6.5), sharey=True)
for ax, h, t in zip(axes, [hist_sim, hist_seq],
                    ["simultaneous update", "sequential update"]):
    ax.imshow(h[:N_SHOWN + 1], aspect="auto", interpolation="nearest",
              cmap="binary")
    ax.set_title(t)
    ax.set_xlabel("position")
axes[0].set_ylabel("time (tick)")
plt.tight_layout()
plt.show()

Look at the two panels side by side. The difference is real but subtle: the
sequential panel has thinner clumps and fewer of the long dark wedges. If you
had only ever seen one of these pictures you would have no reason to suspect
the other existed.

That is the uncomfortable lesson. No line in the three-rule statement of the
model tells you which panel is correct. The choice lives entirely in the
implementation, it is invisible in any verbal description of the model, and it
changes the headline number. This is why journals expect agent-based models to
declare their update scheme, and why "I reimplemented the published model and
got a different answer" is such a common complaint in this literature.

For this particular model there is a defensible answer: simultaneous updating
is the more honest one, because a real driver cannot react in the same instant
that the car ahead moves. The sequential scheme quietly grants every driver an
infinitely fast reaction time, and it is no accident that this is the scheme
that makes the jams go away.

## 10. Extension: giving cars a speed

Everything so far has used one-speed cars, which is what a group can run with a
pencil. The natural next step is to let a car travel several cells per tick,
which turns the model into the **Nagel-Schreckenberg model** — the standard
reference point for this whole family, published in 1992 and still the basis of
a lot of applied traffic simulation.

The rules become four, applied to every car simultaneously:

1. *Accelerate.* Increase your speed by one, up to a maximum.
2. *Avoid collision.* Reduce your speed to the number of empty cells ahead of
   you, if that is smaller.
3. *Randomise.* With probability *p*, reduce your speed by one. This is the old
   brake rule, generalised.
4. *Move.* Advance by your current speed.

Set the maximum speed to one and rules 1, 2 and 4 collapse back into the
worksheet model exactly. Nothing new is being introduced; a constant is being
turned into a variable.

In [ ]:
def nasch(length, n_cars, v_max, p, n_ticks, seed=5):
    """Nagel-Schreckenberg traffic model on a ring.

    This switches representation: instead of a boolean road we keep an array of
    car positions and an array of car speeds. That is because speed is now a
    property that persists between ticks and belongs to a car, so the cars have
    to become first-class objects in the data structure. Representation follows
    from what the agents need to remember.
    """
    rng = np.random.default_rng(seed)
    pos = np.sort(rng.choice(length, size=n_cars, replace=False))
    vel = np.zeros(n_cars, dtype=int)

    history = np.zeros((n_ticks + 1, length), dtype=bool)
    history[0, pos] = True

    for t in range(1, n_ticks + 1):
        # Gap to the car ahead. np.roll pairs each car with its successor, and
        # the modulo handles the car whose successor is round the ring.
        gap = (np.roll(pos, -1) - pos - 1) % length

        vel = np.minimum(vel + 1, v_max)      # rule 1: accelerate
        vel = np.minimum(vel, gap)            # rule 2: do not crash

        slow = rng.random(n_cars) < p         # rule 3: random slowdown
        vel[slow] = np.maximum(vel[slow] - 1, 0)

        pos = (pos + vel) % length            # rule 4: move

        # Cars can overtake nobody in this model, but positions wrap, so we
        # re-sort to keep the "car ahead" relationship correct next tick.
        order = np.argsort(pos)
        pos, vel = pos[order], vel[order]
        history[t, pos] = True

    return history


hist_ns = nasch(length=300, n_cars=60, v_max=3, p=0.25, n_ticks=200)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(hist_ns, aspect="auto", interpolation="nearest", cmap="binary")
ax.set_xlabel("position along the road (cell)")
ax.set_ylabel("time (tick)")
ax.set_title("Nagel-Schreckenberg: 60 cars on 300 cells, "
             "top speed 3, p = 0.25")
plt.tight_layout()
plt.show()

Now the phenomenon is unmistakable. The dense black bands are jams, and they
run from upper right to lower left: they move **backwards** through the road
while every car in the light speckled regions is moving forwards at speed. A
car enters a band from the right, crawls through it, and exits on the left into
open road, whereupon the band has drifted further upstream to swallow the car
behind.

Nothing about this changed the mechanism. Raising the top speed did not create
the backward-propagating jam; it just widened the gap between how fast cars
move and how fast the queue is handed backwards, which is what makes the two
visible as separate features on one plot. The measurement in section 6 was
detecting the same thing all along.

This is a useful thing to have shown students. The minimal model was not wrong,
and the elaborate model was not necessary for the finding. What the elaboration
bought was legibility.

## 11. What this model is and is not

This model contains no cause of jams. There is no obstacle, no accident, no
aggressive driver, no bottleneck, and no coordination. There is a road, a rule
about the cell in front of you, and an occasional pointless tap of the brake.
Out of that comes a structure that outlives the disturbance that created it and
travels in the opposite direction to the traffic.

That is the argument for agent-based modelling in one paragraph. You could not
have reached the backward-travelling jam by reasoning about average speeds, and
you could not have reached the turnover in the flow curve by reasoning about
individual drivers. The behaviour lives in the interaction, so you have to run
the interaction.

**What it leaves out**, at minimum: cars have only one speed, so there is no
acceleration and no speed choice; drivers see exactly one cell ahead, so there
is no anticipation; every driver is identical, so there are no trucks and no
tailgaters; the road is a featureless ring, so there are no lanes, junctions,
or exits. Several of these have been added in the literature and the
backward-propagating jam survives all of them, which is the useful thing to
know about a minimal model — not that it is realistic, but which of its
conclusions are robust to being made more realistic.

**Things worth trying from here**

- Give one car a different rule: it needs two clear cells ahead before moving.
  A single heterogeneous agent on an otherwise uniform road is usually enough
  to reorganise the whole system.
- Let cars see two cells ahead and move only if both are clear. Does
  anticipation help or hurt?
- Run the section 10 model at several densities and rebuild the flow curve.
  Does the peak still sit at density 0.5, and if not, why not?
- Replace the ring with an open road, cars entering at one end at some rate.
  Where does the first jam appear, and why is that answer sensitive to the
  entry rate?
- Break the model on purpose: set the brake probability to 1, or the density to
  1, and check that the output is what the arithmetic says it must be. Models
  that have never been run at their extremes usually have bugs hiding there.